# 🚀 Phase 2: QLoRA Fine-Tuning Llama-3.2-3B for Text-to-SQL
Official 1-Click Fine-Tuning Notebook for **`unsloth/Llama-3.2-3B-Instruct`** on **`b-mc2/sql-create-context`** dataset (78,000+ pairs).

**Hardware Required:** Free Google Colab T4 GPU (Runtime -> Change runtime type -> T4 GPU).

### Step 1: Install Unsloth & Dependencies

In [ ]:
# Install Unsloth package cleanly
!pip install -q unsloth
# Install latest Unsloth nightly & zoo dependencies
!pip install -q --force-reinstall --no-cache-dir --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q datasets mlflow evaluate sacrebleu

### Step 2: Load Base Model in 4-bit Quantization

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto-detect
load_in_4bit = True # 4-bit NF4 quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Attach QLoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### Step 3: Load and Format Text-to-SQL Dataset

In [ ]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert SQL assistant. Given the database schema and a natural language question, write the correct SQL query. Output only the SQL — no explanation, no markdown fences.

### Database Schema (Input):
{}

### Question:
{}

### Response (SQL):
{}"""

EOS_TOKEN = tokenizer.eos_token
def format_prompts(examples):
    contexts  = examples["context"]
    questions = examples["question"]
    answers   = examples["answer"]
    texts = []
    for context, question, answer in zip(contexts, questions, answers):
        text = alpaca_prompt.format(context.strip(), question.strip(), answer.strip()) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("b-mc2/sql-create-context", split = "train")
dataset = dataset.map(format_prompts, batched = True)

### Step 4: Configure Trainer & Run QLoRA Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting QLoRA Fine-Tuning...")
trainer_stats = trainer.train()

### Step 5: Verify Fine-Tuned Model Inference & Save Adapter Weights

In [ ]:
# Enable Fast Inference Mode
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "CREATE TABLE employees (id INT, name TEXT, department TEXT, salary DECIMAL);",
        "Find all employees in Engineering with a salary over 80000",
        "", # Leave answer blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print("--- Generated SQL Result ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Save LoRA adapter weights
model.save_pretrained("adapter_weights")
tokenizer.save_pretrained("adapter_weights")
print("
✅ Adapter weights successfully saved to adapter_weights/")